In [ ]:
import zipfile
with zipfile.ZipFile("/data2/qn/MRAMG/MRAMG-Bench/IMAGE.zip", "r") as z:
    print(z.namelist())

In [2]:
extract_dir = "/data2/qn/MRAMG/MRAMG-Bench/IMAGE"

with zipfile.ZipFile("/data2/qn/MRAMG/MRAMG-Bench/IMAGE.zip", 'r') as z:
    z.extractall(extract_dir)


In [2]:
# 调用bge-m3，生成embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/data2/qn/MRAMG/models/bge-m3")

/data2/qn/anaconda3/envs/kgqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/data2/qn/MRAMG/models/bge-m3")
sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

/data2/qn/anaconda3/envs/kgqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [42]:
from llama_index.core.node_parser import SentenceSplitter
# 初始化切片器
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=0)
import re
def get_chunks_with_metadata(doc, meta_base):
    """
    doc: (doc_id, text, img_list)
    meta_base: 基础元数据，例如 {"source": "manual.pdf"}
    """
    doc_id, text, img_list = doc
    
    # --- 步骤 1: 预处理 - 给 <PIC> 加上索引编号 ---
    # 变成 <PIC_0>, <PIC_1> ... 
    # 由于后面有空格，splitter 几乎百分之百会完整保留这个标签
    counter = 0
    def tag_indexer(match):
        nonlocal counter
        res = f"<PIC_{counter}>"
        counter += 1
        return res
    
    indexed_text = re.sub(r"<PIC>", tag_indexer, text)

    # --- 步骤 2: 切分文本 ---
    # 假设你已经初始化了 splitter = SentenceSplitter(...)
    text_chunks = splitter.split_text(indexed_text)
    
    final_data = []
    for i, chunk in enumerate(text_chunks):
        # --- 步骤 3: 提取当前 chunk 包含的所有图片索引 ---
        # 寻找形如 <PIC_n> 的内容
        found_indices = re.findall(r"<PIC_(\d+)>", chunk)
        
        # 根据索引从 img_list 中捞取对应的图片
        # 即使两个 chunk 有重叠，它们也会根据编号各自捞取正确的图片
        current_chunk_imgs = [img_list[int(idx)] for idx in found_indices]
        
        # --- 步骤 4: 文本还原 ---
        # 将 <PIC_n> 还原为标准 <PIC>，保持数据纯净
        clean_chunk = re.sub(r"<PIC_\d+>", "<PIC>", chunk)
        
        data = {
            "content": clean_chunk,
            "metadata": {
                **meta_base,
                "doc_id": doc_id,
                "imgs_len_of_doc": len(img_list),
                "chunk_index": i,
                "include_img_list": ",".join(map(str, current_chunk_imgs)) # 这里存的是该 chunk 对应的图片(路径/对象)列表
            },
            "embedding": None # 待后续调用 model.encode
        }
        final_data.append(data)
        
    return final_data

In [52]:
import json
jsonl_file = "/data2/qn/MRAMG/MRAMG-Bench/doc_wit.jsonl"
all_chunks = []
with open(jsonl_file, "r") as f:
    for line in f:
        data = json.loads(line)
        all_chunks.extend(get_chunks_with_metadata(data, {"source": "doc_wit"}))


In [7]:
import chromadb
# 1. 初始化本地持久化客户端
# 数据会保存在当前目录下的 'my_local_db' 文件夹中
client = chromadb.PersistentClient(path="/data2/qn/MRAMG/chromadb")

# 2. 创建或获取 Collection（相当于数据库中的表）
# 注意：Chroma 默认使用自带的 embedding 模型，但我们要用自己的 BGE-M3
collection = client.get_or_create_collection(name="doc_arxiv")

In [54]:
ids = [f"id_{i}" for i in range(len(all_chunks))]  
    # 使用你已加载好的 model 生成 embeddings
texts = [c["content"] for c in all_chunks]
embeddings = model.encode(texts)
for i, emb in enumerate(embeddings):
    all_chunks[i]["embedding"] = emb.tolist()

In [55]:
collection.add(
    ids=ids, 
    embeddings=[c["embedding"] for c in all_chunks],
    metadatas=[c["metadata"] for c in all_chunks],
    documents=[c["content"] for c in all_chunks]
)

In [10]:
# 5. 检索示例
query = "How does MVSplat achieve efficient 3D Gaussian prediction and fast inference?"
query_emb = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3,
    # 显式指定需要包含的内容：默认只有 metadatas 和 documents
    include=["documents", "metadatas", "distances"] 
)
print(results)

{'ids': [['id_8', 'id_9', 'id_7']], 'embeddings': None, 'documents': [['4.3.Assessing model efficiency. As reported in Tab. 1, apart from attaining su_x0002_perior image quality, MVSplat also shows the fastest inference time among all the compared models, accompanied by a lightweight model size, demonstrat_x0002_ing its efficiency and practical utility. It is noteworthy that the reported time encompasses both image encoding and rendering stages. For an in-depth time comparison with pixelSplat [1], our encoder runs at 0.043s, which is more than 2× faster than pixelSplat (0.102s). Besides, pixelSplat predicts 3 Gaussians per_x0002_pixel, while our MVSplat predicts 1 single Gaussian, which also contributes to our faster rendering speed (0.0015s vs. 0.0025s) due to the threefold reduction in the number of Gaussians. More importantly, equipped with the cost volume_x0002_based encoder, our MVSplat enables fast feed-forward inference of 3D Gaussians with a much light-weight design, resulting 

In [13]:
results['documents'][0][2]

'Unlike pixelSplat [1] that predicts probabilistic depth, we develop an efficient and high-performance multi-view depth estimation model that enables unprojecting predicted depth maps as the Gaussian centers, in parallel with another branch for prediction of other Gaussian parameters (αj , Σj and cj ). Our full model, illustrated in <PIC>, is trained end-to-end using only a simple rendering loss for supervision. Next, we discuss the key components.The qualitative comparisons of the top three best models are visualized in <PIC>. MVSplat achieves the highest quality on novel view results even un_x0002_der challenging conditions, such as these regions with repeated patterns (“win_x0002_dow frames” in 1st row), or these present in only one of the input views (“stair handrail” and “lampshade” in 2nd and 3rd rows), or when depicting large-scale outdoor objects captured from distant viewpoints (“bridge” in 4th row). The baseline methods exhibit obvious artifacts for these regions, while MVSpl

In [23]:
import re
import json
def get_caption_dict(doc_name, img_list):
    with open(f"/data2/qn/MRAMG/MRAMG-Bench/IMAGE/IMAGE/images_info/{doc_name}_imgs_collection.json") as f:
        data = json.load(f)
    return {img: data[img]['image_caption'] for img in img_list}

def build_prompt_from_chroma(doc_name, chunks):
    
    docs = chunks["documents"][0]
    metas = chunks["metadatas"][0]
    
    context_parts = []
    image_captions = []
    
    global_img_id = 1
    img_path_to_id = {}
    
    for i, (text, meta) in enumerate(zip(docs, metas)):
        
        img_str = meta.get("include_img_list", "")
        
        # 如果没有图片
        if not img_str.strip():
            context_parts.append(f"[context_{i+1}] {text}")
            continue
        
        img_list = img_str.split(",")
        caption_dict = get_caption_dict(doc_name, img_list)
        
        # 逐个替换 <PIC>
        for img_path in img_list:
            
            if img_path not in img_path_to_id:
                img_path_to_id[img_path] = global_img_id
                image_captions.append(f"[img{global_img_id}_caption] {caption_dict[img_path]}")
                global_img_id += 1
            
            img_id = img_path_to_id[img_path]
            
            # 替换一个 <PIC>
            text = text.replace("<PIC>", f"<img{img_id}>", 1)
        
        context_parts.append(f"[context_{i+1}] {text}")
    
    context_block = "\n".join(context_parts)
    caption_block = "\n".join(image_captions)
    
    return context_block, caption_block

In [24]:
build_prompt_from_chroma('arxiv', results)

('[context_1] 4.3.Assessing model efficiency. As reported in Tab. 1, apart from attaining su_x0002_perior image quality, MVSplat also shows the fastest inference time among all the compared models, accompanied by a lightweight model size, demonstrat_x0002_ing its efficiency and practical utility. It is noteworthy that the reported time encompasses both image encoding and rendering stages. For an in-depth time comparison with pixelSplat [1], our encoder runs at 0.043s, which is more than 2× faster than pixelSplat (0.102s). Besides, pixelSplat predicts 3 Gaussians per_x0002_pixel, while our MVSplat predicts 1 single Gaussian, which also contributes to our faster rendering speed (0.0015s vs. 0.0025s) due to the threefold reduction in the number of Gaussians. More importantly, equipped with the cost volume_x0002_based encoder, our MVSplat enables fast feed-forward inference of 3D Gaussians with a much light-weight design, resulting in 10× fewer parameters and more than 2× faster speed comp

In [3]:
prompt = open('prompts/text.txt', 'r').read()
query = "what is your name"
context = "dadaiu dadadad daa"
caption = "['a man is running', 'a man is running']"
llm_input = prompt.format(query=query, context=context, caption=caption[0])


In [4]:
llm_input

'#Input\nQuery: what is your name \nContext: dadaiu dadadad daa    \nImage Caption: [\n\n# Task\nImagine you are an expert in handling multimodal input queries and producing coherent text-image responses. You will receive:\n1. Query: The user query to be answered.\n2. Contexts containing multiple images represented as placeholders <img>.    \n- The input context follows the format: \n[context_1] <img1>, [context_2] <img2>, ...   \n- Each [text_context_x] represents a pure text passage, while each <img> serves as a placeholder for an image.  \n3. A set of image captions.   \n- Each caption is sequentially aligned in a one-to-one correspondence with its respective image placeholder <img>.     \nYour task is to answer the query based solely on the content of the context and input image information. Firstly, you should select appropriate images from the provided context (if none are suitable, you may choose not to include any). And then generate a mixed media response to the query, combini